# Планировщик визуалов для конспектов ЕГЭ

## Презентация результатов разработки и тестирования

**Дата:** 18 марта 2026  
**Задача:** Автоматическое планирование визуализаций для персонализированных конспектов ЕГЭ по математике

---
## 1. Что такое планировщик визуалов?

Планировщик визуалов — ключевой компонент пайплайна генерации учебных материалов.

**Пайплайн:**
```
Конспект (текст) → Планировщик визуалов → Scene JSON → Template Retrieval → Renderer (SVG/TikZ) → Готовая инфографика
```

**Планировщик решает три задачи:**
1. **ГДЕ** в тексте нужен визуал (не каждая секция требует иллюстрации)
2. **КАКОЙ** тип визуализации подходит (из 13 типов в таксономии)
3. **ЧТО** именно должно быть изображено (структурное описание в формате JSON)

---
## 2. Архитектура планировщика

### Таксономия типов визуалов (13 типов):

| Тип | Описание | Пример применения |
|-----|---------|-------------------|
| `function_graph` | График математической функции | Парабола y=x²−4 |
| `table` | Таблица сравнения/классификации | Типы уравнений |
| `step_by_step_diagram` | Пошаговая схема решения | Алгоритм нахождения ОДЗ |
| `outcome_tree` | Дерево возможных исходов | Бросок двух монет |
| `number_line` | Числовая прямая | Знаки производной |
| `coordinate_plane` | Координатная плоскость | Точки пересечения |
| `tangent_line_graph` | График с касательной | Наклон = производная |
| `derivative_sign_chart` | Схема знаков производной | Возрастание/убывание |
| `formula_card` | Карточка с формулой | P = m/n |
| `comparison_chart` | Сравнение концепций | «ровно» vs «хотя бы» |
| `flowchart` | Блок-схема алгоритма | Выбор метода решения |
| `venn_diagram` | Диаграмма Венна | Пересечение событий |
| `annotated_example` | Пример с аннотациями | Решение с выделением шагов |

### Формат ввода/вывода

**Вход:** Markdown-конспект → разбитый на секции по заголовкам

**Выход (для каждой секции):**
```json
{
  "need_visual": true,
  "visual_type": "formula_card",
  "pedagogical_goal": "Помочь запомнить формулу вероятности",
  "description": "Формула P=m/n с пояснениями компонентов",
  "params": {
    "formula_latex": "P = \\frac{m}{n}",
    "components": {"m": "благоприятные исходы", "n": "все исходы"}
  },
  "caption": "Формула классической вероятности",
  "placement": "after_section",
  "priority": "high"
}
```

In [ ]:
import json
from pathlib import Path
from collections import Counter

RESULTS_DIR = Path("results")

with open(RESULTS_DIR / "all_results.json", "r") as f:
    all_results = json.load(f)

MODELS = list(all_results.keys())
TASKS = ["task4", "task6", "task8", "task12"]
TASK_LABELS = {
    "task4": "№4 Классическая вероятность",
    "task6": "№6 Уравнения",
    "task8": "№8 Производная и графики",
    "task12": "№12 Исследование функций",
}
MODEL_SHORT = {m: m.split('/')[-1] for m in MODELS}

print("Загружены результаты для:")
for m in MODELS:
    print(f"  • {MODEL_SHORT[m]}")
print(f"\nКонспекты: {', '.join(TASK_LABELS.values())}")

---
## 3. Сравнение моделей: количественные метрики

In [ ]:
# Сводная таблица: количество визуалов / секций / время / токены
print(f"{'Модель':<40} {'Конспект':<32} {'Визуалы':>8} {'Секции':>8} {'Время,с':>8} {'Токены':>10}")
print("=" * 110)

model_totals = {}

for model in MODELS:
    m_short = MODEL_SHORT[model]
    model_totals[model] = {"visuals": 0, "sections": 0, "time": 0, "tokens_in": 0, "tokens_out": 0}
    
    for task in TASKS:
        r = all_results[model].get(task, {})
        if "error" in r:
            print(f"{m_short:<40} {TASK_LABELS[task]:<32} {'ERROR':>8}")
            continue
        
        vis = r["visuals_planned"]
        sec = r["sections_analyzed"]
        t = r["total_time_sec"]
        tok = r["total_tokens_in"] + r["total_tokens_out"]
        
        model_totals[model]["visuals"] += vis
        model_totals[model]["sections"] += sec
        model_totals[model]["time"] += t
        model_totals[model]["tokens_in"] += r["total_tokens_in"]
        model_totals[model]["tokens_out"] += r["total_tokens_out"]
        
        print(f"{m_short:<40} {TASK_LABELS[task]:<32} {vis:>5}/{sec:<2} {sec:>8} {t:>8.1f} {tok:>10,}")
    print()

In [ ]:
# Итоговое сравнение моделей
print("\n" + "=" * 90)
print("ИТОГО ПО МОДЕЛЯМ (все 4 конспекта)")
print("=" * 90)
print(f"{'Модель':<40} {'Визуалов':>10} {'Секций':>8} {'Время,с':>8} {'Токены вх.':>12} {'Токены вых.':>12}")
print("-" * 90)

for model in MODELS:
    mt = model_totals[model]
    m_short = MODEL_SHORT[model]
    pct = (mt['visuals'] / mt['sections'] * 100) if mt['sections'] > 0 else 0
    print(f"{m_short:<40} {mt['visuals']:>7} ({pct:.0f}%) {mt['sections']:>6} {mt['time']:>8.1f} {mt['tokens_in']:>12,} {mt['tokens_out']:>12,}")

---
## 4. Распределение типов визуалов по моделям

In [ ]:
# Распределение типов визуалов
for model in MODELS:
    m_short = MODEL_SHORT[model]
    print(f"\n{'='*60}")
    print(f"  {m_short}")
    print(f"{'='*60}")
    
    all_types = Counter()
    for task in TASKS:
        r = all_results[model].get(task, {})
        if "error" in r:
            continue
        for p in r["plans"]:
            if p["plan"]["need_visual"]:
                all_types[p["plan"]["visual_type"]] += 1
    
    total = sum(all_types.values())
    for vtype, count in all_types.most_common():
        bar = "█" * (count * 2)
        pct = count / total * 100 if total > 0 else 0
        print(f"  {vtype:<28} {count:>3} ({pct:>5.1f}%)  {bar}")

---
## 5. Примеры запланированных визуалов (качественный анализ)

In [ ]:
# Показываем лучшие примеры визуалов от каждой модели
def show_visual_plan(plan_entry, model_name):
    p = plan_entry["plan"]
    if not p["need_visual"]:
        return
    print(f"  ┌─ Секция: {p['section_title']}")
    print(f"  │  Тип: {p['visual_type']}")
    print(f"  │  Цель: {p['pedagogical_goal'][:120]}")
    print(f"  │  Описание: {p['description'][:150]}")
    if p['params']:
        params_str = json.dumps(p['params'], ensure_ascii=False, indent=4)
        for line in params_str.split('\n')[:8]:
            print(f"  │  {line}")
        if len(params_str.split('\n')) > 8:
            print(f"  │  ...")
    print(f"  │  Подпись: {p['caption']}")
    print(f"  └─ Приоритет: {p['priority']}")
    print()

# Task 4 (вероятность) — хорошо показывает outcome_tree и formula_card
print("\n" + "=" * 70)
print("ПРИМЕРЫ: №4 Классическая вероятность (Qwen 2.5-7B)")
print("=" * 70)
task4_plans = all_results[MODELS[0]]["task4"]["plans"]
for p in task4_plans:
    if p["plan"]["need_visual"] and p["plan"]["priority"] == "high":
        show_visual_plan(p, MODELS[0])

In [ ]:
# Task 8 (производная) — показывает tangent_line_graph и derivative_sign_chart
print("\n" + "=" * 70)
print("ПРИМЕРЫ: №8 Производная и графики (Qwen 2.5-7B)")
print("=" * 70)
task8_plans = all_results[MODELS[0]]["task8"]["plans"]
for p in task8_plans:
    if p["plan"]["need_visual"]:
        show_visual_plan(p, MODELS[0])

In [ ]:
# Task 12 (исследование функций) — derivative_sign_chart, function_graph
print("\n" + "=" * 70)
print("ПРИМЕРЫ: №12 Исследование функций (Mistral Small 24B)")
print("=" * 70)
task12_plans = all_results[MODELS[1]]["task12"]["plans"]
for p in task12_plans:
    if p["plan"]["need_visual"] and p["plan"]["visual_type"] in ["derivative_sign_chart", "function_graph", "step_by_step_diagram"]:
        show_visual_plan(p, MODELS[1])

---
## 6. Сравнение поведения моделей на одной и той же секции

In [ ]:
# Сравним как разные модели оценивают одни и те же секции
def compare_section(task, section_idx):
    """Сравнить решения всех моделей для конкретной секции."""
    for model in MODELS:
        m_short = MODEL_SHORT[model]
        r = all_results[model].get(task, {})
        if "error" in r:
            print(f"  {m_short}: ERROR")
            continue
        plans = r["plans"]
        if section_idx >= len(plans):
            print(f"  {m_short}: секция {section_idx} не найдена")
            continue
        p = plans[section_idx]["plan"]
        if p["need_visual"]:
            print(f"  {m_short:<35} → {p['visual_type']:<25} приоритет: {p['priority']:<8} цель: {p['pedagogical_goal'][:80]}")
        else:
            print(f"  {m_short:<35} → визуал не нужен")

# Секция с формулой вероятности (task4)
print("\n" + "=" * 120)
print("Секция: 'Формула классической вероятности' (task4)")
print("=" * 120)
compare_section("task4", 3)  # formula section

print("\n" + "=" * 120)
print("Секция: 'Пример 1: Бросок двух монет' (task4)")
print("=" * 120)
compare_section("task4", 5)  # example 1

print("\n" + "=" * 120)
print("Секция: 'Найти значение производной по наклону касательной' (task8)")
print("=" * 120)
compare_section("task8", 5)  # algorithm

print("\n" + "=" * 120)
print("Секция: 'Пример 1: Найти наибольшее значение функции' (task12)")
print("=" * 120)
compare_section("task12", 8)  # example

---
## 7. Оценка качества JSON-выхода

In [ ]:
# Проверяем качество структурированного вывода
print(f"{'Модель':<40} {'JSON ok':>8} {'JSON fail':>10} {'% успеха':>10} {'Ср. токены/секцию':>18}")
print("=" * 90)

for model in MODELS:
    m_short = MODEL_SHORT[model]
    json_ok = 0
    json_fail = 0
    total_tokens = 0
    total_sections = 0
    
    for task in TASKS:
        r = all_results[model].get(task, {})
        if "error" in r:
            continue
        for p in r["plans"]:
            total_sections += 1
            total_tokens += p["llm_meta"]["tokens_out"]
            raw = p["llm_meta"]["raw_response"]
            try:
                # Check if we successfully parsed JSON
                if p["plan"]["visual_type"] != "" or not p["plan"]["need_visual"]:
                    json_ok += 1
                else:
                    json_fail += 1
            except:
                json_fail += 1
    
    pct = json_ok / (json_ok + json_fail) * 100 if (json_ok + json_fail) > 0 else 0
    avg_tok = total_tokens / total_sections if total_sections > 0 else 0
    print(f"{m_short:<40} {json_ok:>8} {json_fail:>10} {pct:>9.1f}% {avg_tok:>17.0f}")

---
## 8. Полный визуальный план для одного конспекта

Ниже — полный план визуалов для конспекта задания №4 (Классическая вероятность), сгенерированный моделью Qwen 2.5-7B. Это то, что передаётся на вход рендереру.

In [ ]:
# Полный план визуалов для task4 (Qwen)
result = all_results[MODELS[0]]["task4"]

print(f"Конспект: №4 Классическая вероятность")
print(f"Модель: {MODEL_SHORT[MODELS[0]]}")
print(f"Всего секций: {result['total_sections']}")
print(f"Проанализировано: {result['sections_analyzed']}")
print(f"Визуалов запланировано: {result['visuals_planned']}")
print(f"Время: {result['total_time_sec']}s")
print()

for i, entry in enumerate(result["plans"]):
    p = entry["plan"]
    status = "🟢 ВИЗУАЛ" if p["need_visual"] else "⚪ нет"
    vtype = p["visual_type"] if p["need_visual"] else "-"
    pri = p["priority"] if p["need_visual"] else "-"
    print(f"  [{p['section_id']:>2}] {status} {p['section_title'][:50]:<52} тип: {vtype:<25} приоритет: {pri}")

---
## 9. Пример полного Scene JSON (готов для рендерера)

In [ ]:
# Покажем 3 наиболее интересных visual plan как Scene JSON
examples = [
    ("task4", 0, "Qwen — outcome_tree для вероятности"),
    ("task8", 0, "Qwen — tangent_line_graph для производной"),
    ("task12", 1, "Mistral — derivative_sign_chart для экстремумов"),
]

for task, model_idx, label in examples:
    model = MODELS[model_idx]
    r = all_results[model].get(task, {})
    if "error" in r:
        continue
    visual_plans = [p for p in r["plans"] if p["plan"]["need_visual"]]
    if visual_plans:
        best = visual_plans[0]
        print(f"\n{'='*70}")
        print(f"  {label}")
        print(f"{'='*70}")
        scene_json = {
            "visual_type": best["plan"]["visual_type"],
            "pedagogical_goal": best["plan"]["pedagogical_goal"],
            "description": best["plan"]["description"],
            "params": best["plan"]["params"],
            "caption": best["plan"]["caption"],
            "placement": best["plan"]["placement"],
        }
        print(json.dumps(scene_json, ensure_ascii=False, indent=2))

---
## 10. Выводы и рекомендации

In [ ]:
# Финальная сводка
print("""╔══════════════════════════════════════════════════════════════════════════╗
║                     ВЫВОДЫ И РЕКОМЕНДАЦИИ                               ║
╚══════════════════════════════════════════════════════════════════════════╝

1. ПЛАНИРОВЩИК РАБОТАЕТ
   Все три модели успешно анализируют конспекты и генерируют
   структурированные планы визуалов в формате JSON.

2. СРАВНЕНИЕ МОДЕЛЕЙ:""")

for model in MODELS:
    mt = model_totals[model]
    m_short = MODEL_SHORT[model]
    pct = (mt['visuals'] / mt['sections'] * 100) if mt['sections'] > 0 else 0
    avg_time = mt['time'] / 4  # 4 tasks
    print(f"\n   • {m_short}:")
    print(f"     - Визуалов: {mt['visuals']} из {mt['sections']} секций ({pct:.0f}%)")
    print(f"     - Среднее время на конспект: {avg_time:.0f}s")
    print(f"     - Токены: {mt['tokens_in'] + mt['tokens_out']:,}")

print("""  
3. РЕКОМЕНДАЦИЯ ПО МОДЕЛИ:
   → Qwen 2.5-7B-Instruct-Turbo — лучший баланс скорости, качества и стоимости.
     Генерирует адекватное количество визуалов (не перегружает),
     корректно определяет типы, даёт содержательные params.
   → Mistral Small 24B — генерирует больше визуалов, разнообразнее типы,
     но может быть избыточным (почти каждая секция → визуал).
   → Gemma 3n-E4B — хороший средний вариант, но медленнее.

4. КАЧЕСТВО ВЫХОДА:
   - JSON-парсинг: ~100% успех у всех моделей
   - Типы визуалов: контекстно адекватны тематике конспекта
   - Params: содержат конкретные уравнения, точки, шаги
   - Готовы для передачи в renderer (SVG/TikZ)

5. СЛЕДУЮЩИЕ ШАГИ:
   a) Разработать SVG-рендерер для каждого типа визуала
   b) Создать библиотеку шаблонов (template retrieval)
   c) Добавить персонализацию (уровень ученика → сложность визуала)
   d) Интегрировать в основной пайплайн генерации конспектов
   e) A/B тестирование с реальными учениками
""")